In [0]:
dbutils.widgets.removeAll()

In [0]:
# Set input parameters - point to external JSON config file
clientContainer = "claimsprocessing" 
subGroupConfigPath = "/Workspace/Repos/logi@openhealthagents.org/alphaesai/ClaimsProcessing/DimMember/Gold/Schema/dimMember.json"

print(f"Client Container: {clientContainer}")
print(f"Config Path: {subGroupConfigPath}")


In [0]:
# Setup widgets (if not already defined)
dbutils.widgets.text("ClientContainer", "", "Client Container")
dbutils.widgets.text("SubGroupConfigPath", "", "SubGroup Config Path")

# Get widget values, but use pre-defined values if widgets are empty
widget_client = dbutils.widgets.get("ClientContainer")
widget_config = dbutils.widgets.get("SubGroupConfigPath")

# Use widget values if provided, otherwise use pre-defined variables
if widget_client:
    clientContainer = widget_client
if widget_config:
    subGroupConfigPath = widget_config

In [0]:
from pyspark.sql.functions import explode, col
import os
import json

In [0]:
# Helper function to check if path exists
def path_exists(path):
    """Check if a path exists (table, file, or directory)"""
    try:
        # Check if it's a table path (catalog.schema.table format)
        if '.' in path and not path.startswith('/'):
            parts = path.split('.')
            if len(parts) == 3:
                # It's a table reference
                return spark.catalog.tableExists(path)
        
        # Check if it's a file/directory path
        if path.startswith('/'):
            try:
                dbutils.fs.ls(path)
                return True
            except:
                return False
        
        # Check as Unity Catalog table
        return spark.catalog.tableExists(path)
    except:
        return False

In [0]:
# Read JSON configuration file
print(f"Reading config from: {subGroupConfigPath}")

# Read JSON file from Workspace using Python
with open(subGroupConfigPath, 'r') as f:
    config_data = json.load(f)

# Extract SubLayerProcessing array - work with Python dict, not DataFrame
subLayerProcessing = config_data.get("SubLayerProcessing", [])

print(f"Found {len(subLayerProcessing)} sub-group entities to process")
for entity in subLayerProcessing:
    print(f"  - {entity.get('SubGroupEntity', 'Unknown')}")

# Process each sub-group entity
processing_results = []

for entity_row in subLayerProcessing:
    subGroupEntity = entity_row.get("SubGroupEntity", "Unknown")
    print(f"\n{'='*60}")
    print(f"Processing Entity: {subGroupEntity}")
    print(f"{'='*60}")
    
    # Flag to determine if all sources are valid
    all_sources_valid = True
    
    # Get destination table path (replace #clientCode placeholder)
    destinationTable = entity_row.get("DestinationTable", "").replace("#clientCode", clientContainer)
    print(f"Destination: {destinationTable}")
    
    # Check if destination exists
    dest_exists = path_exists(destinationTable)
    if not dest_exists:
        print(f"WARNING: Destination table {destinationTable} does not exist yet")
    
    # Get source tables for this entity
    source_tables = entity_row.get("SourceTables", [])
    
    # Load each source table as temp view
    for source_row in source_tables:
        entity_name = source_row.get("Entity", "")
        source_table = source_row.get("SourceTable", "").replace("#clientCode", clientContainer)
        source_format = source_row.get("SourceFormat", "delta")
        
        print(f"  Loading source: {entity_name} from {source_table}")
        
        # Check if source exists
        if path_exists(source_table):
            try:
                # Detect Unity Catalog table vs file path
                if '.' in source_table and not source_table.startswith('/'):
                    # Unity Catalog table: pharma_catalog.gold.member
                    df_file = spark.table(source_table)
                else:
                    # File/mount path
                    df_file = spark.read.format(source_format).load(source_table)
                
                # Create temp view
                df_file.createOrReplaceTempView(entity_name)
                row_count = df_file.count()
                print(f"  {entity_name} temp view created ({row_count} rows)")
            except Exception as e:
                all_sources_valid = False
                print(f"  Failed to load {entity_name}: {str(e)}")
        else:
            all_sources_valid = False
            print(f"  Path for {entity_name} does not exist: {source_table}")
    
    # If all sources are valid, run the SQL and merge scripts
    if all_sources_valid:
        try:
            # 1. Read Processing SQL from external file path defined in JSON config
            sql_script_path = entity_row.get("SQLScriptPath", "")
            print(f"\n  Reading SQL Script from: {sql_script_path}")
            with open(sql_script_path, 'r') as sql_file:
                mainSQLQuery = sql_file.read()
            
            print(f"  Executing SQL Script...")
            mDF = spark.sql(mainSQLQuery)
            mDF.createOrReplaceTempView("tempSQLScript")
            row_count = mDF.count()
            print(f"  tempSQLScript temp view created with {row_count} rows")
            
            # Check if destination table exists, create if not
            print(f"\n  Checking destination table...")
            if not dest_exists:
                # Extract catalog and schema from destination table
                table_parts = destinationTable.split('.')
                if len(table_parts) == 3:
                    catalog, schema, table = table_parts
                    
                    # Create schema if it doesn't exist
                    print(f"  Ensuring schema exists: {catalog}.{schema}")
                    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
                
                print(f"  Creating new destination table: {destinationTable}")
                # Create the destination table with schema from tempSQLScript
                mDF.limit(0).write.format("delta").mode("append").saveAsTable(destinationTable)
            
            # Load destination table as temp view for MERGE mapping
            spark.table(destinationTable).createOrReplaceTempView("DestinationTable")
            print(f"  DestinationTable temp view created")
            
            # 2. Read Merge SQL from external file path defined in JSON config
            merge_script_path = entity_row.get("MergeScriptPath", "")
            print(f"\n  Reading Merge Script from: {merge_script_path}")
            with open(merge_script_path, 'r') as merge_file:
                raw_merge_query = merge_file.read()
                
            print(f"  Executing Merge Script...")
            # Replaces the generic text 'DestinationTable' in the file with the true Unity Catalog identifier
            mergeScript = raw_merge_query.replace("DestinationTable", destinationTable)
            spark.sql(mergeScript)
            print(f" Merge completed successfully for {subGroupEntity}")
            
            processing_results.append({
                "entity": subGroupEntity,
                "status": "SUCCESS",
                "destination": destinationTable,
                "error": ""
            })
            
        except Exception as e:
            error_msg = f"Failed to execute SQL/Merge for {subGroupEntity}: {str(e)}"
            print(f"  {error_msg}")
            processing_results.append({
                "entity": subGroupEntity,
                "status": "FAILED",
                "destination": destinationTable,
                "error": str(e)
            })
    else:
        error_msg = "One or more source paths do not exist"
        print(f"  Skipping {subGroupEntity}: {error_msg}")
        processing_results.append({
            "entity": subGroupEntity,
            "status": "SKIPPED",
            "destination": destinationTable,
            "error": error_msg
        })

In [0]:
dfgold = spark.read.table("claimsprocessing.gold.gold_dimmember")
display(dfgold)

In [0]:
# Create summary DataFrame
print(f"\n{'='*60}")
print("Processing Summary")
print(f"{'='*60}")

df_results = spark.createDataFrame(processing_results)
display(df_results)

# Build return message
success_count = len([r for r in processing_results if r["status"] == "SUCCESS"])
failed_count = len([r for r in processing_results if r["status"] == "FAILED"])
skipped_count = len([r for r in processing_results if r["status"] == "SKIPPED"])

return_message = f"Processed {len(processing_results)} entities: {success_count} succeeded, {failed_count} failed, {skipped_count} skipped"
print(f"\n{return_message}")

dbutils.notebook.exit(return_message)

In [0]:
df = spark.read.table("claimsprocessing.silver.member")
display(df)

In [0]:
dfgold = spark.read.table("claimsprocessing.gold.gold_dimmember")
display(dfgold)

In [0]:
dfsilver = spark.read.table("claimsprocessing.silver.member")
display(len(dfsilver.columns))

In [0]:
consolidated_member = spark.read.table("claimsprocessing.bronze.member_consolidated")
display(consolidated_member)

In [0]:
dfgold = spark.read.table("claimsprocessing.gold.gold_dimmember")
display(len(dfgold.columns))

print("columns: ")
dfgold.printSchema()